# Visualizacion de casos -- eje, pipeline sin malla

Selecciona objetos con error angular BAJO / MEDIO / ALTO (segun
`eval_..._triangulation_results.json`, ya generado por `Mapping/evaluate.py`)
y para cada uno muestra:

1. **2D**: los renders usados, con los 2 puntos que devolvio Molmo2 superpuestos.
2. **3D**: la malla real, el eje GT, el eje predicho (metodo `triangulation`,
   sin malla), y los rayos camara-punto de cada vista usada -- para ver a
   ojo si la triangulacion se ve razonable o si hay vistas claramente
   inconsistentes entre si.

No hace falta GPU ni re-correr nada -- todo se lee de archivos ya generados
(`molmo_multiview_<EXP>.json`, `predicted_symmetry_<EXP>.json`,
`eval_..._<EXP>_triangulation_results.json`, la malla `.obj` y el `.txt` de
ground truth). Complementa a `docs/diagnostico_conditioning_axis.md` -- las
correlaciones de esos scripts dicen QUE tan fuerte es un efecto en agregado;
este notebook deja ver, caso por caso, COMO se ve un objeto que salio bien
vs. uno que salio mal.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

REPO_ROOT    = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
DATA_ROOT    = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\data")
RENDERS_ROOT = DATA_ROOT / "renders"
OBJECTS_ROOT = DATA_ROOT / "objects"

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "Mapping"))

from pipeline_common.datasets import OBJECTS_SUBDIR, load_mesh
from pipeline_common.naming import exp_filename
from pipeline_common.triangulation import ray_dir_for_point
from evaluate import parse_true_label

# --- Parametros de la corrida a inspeccionar ---
EXPERIMENT_ID = "axis_v06_nomesh"
N_VIEWS       = 26
SIZE, LIGHTING = 224, "flat"
N_PER_BUCKET  = 3          # cuantos objetos mostrar por franja (bueno/medio/malo)
MAX_VIEWS_2D  = 6          # limite de imagenes a mostrar por objeto (evita saturar el notebook)
SAVE_DIR      = None       # Path("../results/diagnostics/plots_casos") para guardar en vez de mostrar

plt.rcParams.update({"figure.dpi": 110})

## 1. Seleccion de casos por franja de error

In [ ]:
eval_json_path = (
    RENDERS_ROOT / "axis_sym"
    / f"eval_s{SIZE}_{LIGHTING}_{EXPERIMENT_ID}_triangulation_results.json"
)
assert eval_json_path.exists(), f"No encontrado: {eval_json_path}"

with open(eval_json_path, encoding="utf-8") as f:
    eval_data = json.load(f)

nv_key = str(N_VIEWS)
ok_objects = []
for obj_id, per_nv in eval_data.get("objects", {}).items():
    if per_nv is None or nv_key not in per_nv:
        continue
    m = per_nv[nv_key]
    if isinstance(m, dict) and m.get("status") == "ok":
        ok_objects.append((obj_id, m["angular_error_deg"]))

ok_objects.sort(key=lambda t: t[1])
n = len(ok_objects)
print(f"{n} objetos con prediccion valida en n_views={N_VIEWS} para {EXPERIMENT_ID}")

buenos = ok_objects[:N_PER_BUCKET]
malos  = ok_objects[-N_PER_BUCKET:][::-1]
mid_start = n // 2 - N_PER_BUCKET // 2
medios = ok_objects[mid_start: mid_start + N_PER_BUCKET]

casos = [("BUENO", o, e) for o, e in buenos] + \
        [("MEDIO", o, e) for o, e in medios] + \
        [("MALO",  o, e) for o, e in malos]

print("\nCasos seleccionados:")
for bucket, obj_id, err in casos:
    print(f"  [{bucket:5s}] {obj_id}  angular_error={err:.2f} grados")

## 2. Helpers de carga y ploteo

In [ ]:
def load_case_data(object_id: str):
    """Carga todo lo necesario para un objeto: molmo (puntos 2D + camaras),
    eje predicho (triangulation), eje GT, y la malla real."""
    obj_dir    = RENDERS_ROOT / "axis_sym" / object_id
    render_dir = obj_dir / str(SIZE) / LIGHTING

    molmo_path = render_dir / exp_filename("molmo_multiview.json", EXPERIMENT_ID)
    with open(molmo_path, encoding="utf-8") as f:
        molmo_data = json.load(f)
    group = molmo_data[str(N_VIEWS)]
    images_sent     = group["images_sent"]
    points_by_image = group["points_by_image"]

    manifest_path = render_dir / "manifest.json"
    manifest = json.load(open(manifest_path, encoding="utf-8")) if manifest_path.exists() else {}
    image_size = manifest.get("image_size", SIZE)
    fov_deg    = manifest.get("fov", 60.0)

    pred_path = obj_dir / exp_filename("predicted_symmetry.json", EXPERIMENT_ID)
    with open(pred_path, encoding="utf-8") as f:
        pred_data = json.load(f)
    pred = pred_data["n_views_predictions"][str(N_VIEWS)]["triangulation"]
    pred_origin    = np.array(pred["origin"])
    pred_direction = np.array(pred["direction"])

    txt_path = OBJECTS_ROOT / OBJECTS_SUBDIR["axis_sym"] / f"{object_id}.txt"
    gt = parse_true_label(txt_path)
    gt_origin    = np.array(gt["elements"][0]["origin"])
    gt_direction = np.array(gt["elements"][0]["direction"])

    obj_path = OBJECTS_ROOT / OBJECTS_SUBDIR["axis_sym"] / f"{object_id}.obj"
    mesh = load_mesh(obj_path)

    return {
        "render_dir": render_dir, "images_sent": images_sent,
        "points_by_image": points_by_image,
        "image_size": image_size, "fov_deg": fov_deg,
        "pred_origin": pred_origin, "pred_direction": pred_direction,
        "gt_origin": gt_origin, "gt_direction": gt_direction,
        "mesh": mesh,
    }


def molmo_xy_to_pixels(x: float, y: float, img_w: int, img_h: int) -> tuple[float, float]:
    """Los x/y que devuelve Molmo2 estan en escala 0-1000 (esquina superior
    izquierda como origen) -- pipeline_common/camera.py::molmo_to_ndc. Para
    graficarlos sobre el PNG real hay que reescalarlos al tamano real de la
    imagen cargada, NO usarlos como pixeles crudos."""
    return (x / 1000.0) * img_w, (y / 1000.0) * img_h


def plot_2d_views(case: dict, object_id: str, bucket: str, err: float) -> None:
    """Muestra hasta MAX_VIEWS_2D renders con los 2 puntos de Molmo2 superpuestos.
    Si el PNG no esta disponible localmente, muestra un placeholder con las
    coordenadas (para poder usarse tambien sin los renders sincronizados)."""
    idxs = sorted(case["points_by_image"].keys(), key=int)[:MAX_VIEWS_2D]
    n = len(idxs)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.4))
    if n == 1:
        axes = [axes]

    for ax, idx_str in zip(axes, idxs):
        cam = case["images_sent"][int(idx_str)]
        img_path = case["render_dir"] / cam["filename"]
        pts = case["points_by_image"][idx_str]

        if img_path.exists():
            img = plt.imread(img_path)
            ax.imshow(img)
            img_h, img_w = img.shape[0], img.shape[1]
        else:
            img_w = img_h = case["image_size"]
            ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
            ax.set_facecolor("#eee")
            ax.text(0.5, 0.5, "(render no\nencontrado)", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="gray")

        for p in pts:
            px, py = molmo_xy_to_pixels(p["x"], p["y"], img_w, img_h)
            color = "red" if p["obj_id"] == 1 else "blue"
            ax.scatter([px], [py], c=color, s=60, edgecolors="white", linewidths=1.2, zorder=5)
        ax.set_title(f"img {idx_str}  (az={cam.get('azimuth','?')}, el={cam.get('elevation','?')})", fontsize=8)
        ax.axis("off")

    fig.suptitle(f"[{bucket}] {object_id}  --  angular_error={err:.2f} grados  "
                 f"(rojo=obj_id 1, azul=obj_id 2)", fontsize=10)
    fig.tight_layout()
    _save_or_show(fig, f"{bucket}_{object_id}_2d.png")


def plot_3d_case(case: dict, object_id: str, bucket: str, err: float) -> None:
    """Malla + eje GT + eje predicho + rayos camara-punto de cada vista usada."""
    mesh = case["mesh"]
    verts = np.asarray(mesh.vertices)
    if len(verts) > 4000:  # subsample para que el plot no sea pesadisimo
        idx = np.random.default_rng(0).choice(len(verts), 4000, replace=False)
        verts_plot = verts[idx]
    else:
        verts_plot = verts

    bbox_diag = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))
    half_len  = bbox_diag * 0.75

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(verts_plot[:, 0], verts_plot[:, 1], verts_plot[:, 2],
               s=1, c="lightgray", alpha=0.4, label="malla (vertices)")

    def plot_line(origin, direction, color, label):
        direction = direction / np.linalg.norm(direction)
        p0 = origin - direction * half_len
        p1 = origin + direction * half_len
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]],
                color=color, linewidth=2.5, label=label)

    plot_line(case["gt_origin"], case["gt_direction"], "green", "eje GT")
    plot_line(case["pred_origin"], case["pred_direction"], "red", "eje predicho (sin malla)")

    # Rayos camara-punto de cada vista usada (grises, cortos, solo para contexto visual)
    # ray_dir_for_point ya hace la conversion 0-1000 -> NDC internamente (via molmo_to_ndc),
    # asi que aca SI se le pasan p["x"]/p["y"] crudos -- no reescalar de nuevo.
    ray_len = bbox_diag * 1.2
    for idx_str, pts in case["points_by_image"].items():
        cam = case["images_sent"][int(idx_str)]
        for p in pts:
            C, d = ray_dir_for_point(p["x"], p["y"], cam["R"], cam["T"], case["fov_deg"], case["image_size"])
            end = C + d * ray_len
            ax.plot([C[0], end[0]], [C[1], end[1]], [C[2], end[2]],
                    color="steelblue", alpha=0.25, linewidth=0.6)

    ax.set_title(f"[{bucket}] {object_id}  --  angular_error={err:.2f} grados", fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_box_aspect([1, 1, 1])
    fig.tight_layout()
    _save_or_show(fig, f"{bucket}_{object_id}_3d.png")


def _save_or_show(fig, filename: str) -> None:
    if SAVE_DIR is not None:
        save_dir = Path(SAVE_DIR)  # acepta str o Path, evita el AttributeError si se seteo como string
        save_dir.mkdir(parents=True, exist_ok=True)
        out = save_dir / filename
        fig.savefig(out, dpi=130, bbox_inches="tight")
        print(f"Guardado: {out}")
        plt.close(fig)
    else:
        plt.show()

## 3. Recorrer los casos seleccionados

In [ ]:
for bucket, object_id, err in casos:
    print("=" * 90)
    print(f"[{bucket}] {object_id}  --  angular_error = {err:.2f} grados  (n_views={N_VIEWS})")
    print("=" * 90)
    try:
        case = load_case_data(object_id)
    except Exception as e:
        print(f"  [error cargando el caso] {e}")
        continue

    plot_2d_views(case, object_id, bucket, err)
    plot_3d_case(case, object_id, bucket, err)
    print()

## 4. Notas de lectura

- **2D**: si el punto rojo (obj_id 1) y azul (obj_id 2) aparecen sobre
  estructuras FISICAMENTE distintas entre vistas de un mismo objeto (p.ej.
  "polo superior" en una vista pero "un lateral" en otra), es evidencia
  visual directa de inconsistencia de identidad entre vistas -- la hipotesis
  que quedo como mas plausible en `docs/diagnostico_conditioning_axis.md`.
- **2D, vistas con az/el extremos**: revisa si las vistas con elevacion muy
  alta/baja (cercanas a +-90) son las que muestran puntos mas ambiguos --
  conecta con `Mapping/diagnose_view_geometry.py` (vistas 'de punta'
  respecto al eje GT).
- **3D**: si los rayos grises (camara-punto) convergen razonablemente cerca
  de la linea roja (eje predicho) pero la linea roja esta lejos/rotada
  respecto a la verde (GT), el sistema esta "consistente consigo mismo" pero
  equivocado -- otra vez, sesgo (identidad) mas que ruido (conditioning).
  Si los rayos grises estan dispersos y no convergen ni siquiera entre si,
  el problema es de dispersion/ruido bruto en la localizacion 2D.